<div style="max-width:900px; margin:40px auto 30px auto; padding:30px 24px;
            text-align:center; font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:15px; font-weight:600; letter-spacing:0.18em;
              text-transform:uppercase; color:#555; margin-bottom:8px;">
    EPIC Jr II 2026
  </div>

  <div style="font-size:18px; letter-spacing:0.08em;
              color:#777; margin-bottom:26px;">
    Hands-On Astro
  </div>

  <div style="width:70px; height:2px; background:#333;
              margin:0 auto 26px auto;"></div>

  <div style="font-size:42px; font-weight:700; line-height:1.15;
              color:#222; margin-bottom:50px;">
    Dos cúmulos, dos relojes
  </div>

  <div style="font-size:13px; letter-spacing:0.10em;
              text-transform:uppercase; color:#888; margin-bottom:6px;">
    Autores del Proyecto
  </div>

  <div style="font-size:16px; color:#444; line-height:1.7;">
    Pablo Escobar (Yachay Tech) &nbsp;&middot;&nbsp; Ariana Guerrón (USFQ)
  </div>

</div>

<div style="width:72%; margin:55px 0 30px 0; padding:3px 0 14px 18px;
            font-family:Arial, Helvetica, sans-serif;
            border-left:3px solid #526b84;">

  <div style="font-size:15px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    Sesión 3
  </div>

  <div style="font-size:30px; font-weight:650; line-height:1.25;
              color:#222; margin-bottom:6px;">
    Leamos los relojes estelares

  </div>

  <div style="font-size:15px; color:#666; line-height:1.5;">
    ¿ Qué cúmulo es más antiguo y qué evidencia lo demuestra?

  </div>

</div>

## Preparación en Google Colab

1. Abran el notebook en Google Colab.
2. Ejecuten las celdas en orden con el botón **▶**.
3. Cuando se solicite, suban `two_clusters.csv`.

No necesitan instalar paquetes ni consultar catálogos astronómicos.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 14, "axes.labelsize": 12, "font.size": 11,})

CLUSTERS = ["Pleiades", "Messier 67"]
LABELS = {"Pleiades": "Pléyades", "Messier 67": "Messier 67"}
COLORS = {"Pleiades": "#2F80ED", "Messier 67": "#F2994A"}

print("Herramientas preparadas.")

In [ ]:
# Buscar el CSV y, si no está disponible, abrir el cargador de Colab.
candidate_paths = [
    Path("Project_B/student_data/two_clusters.csv"),
    Path("student_data/two_clusters.csv"),
    Path("two_clusters.csv"),
    Path("/content/two_clusters.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    try:
        from google.colab import files
        print("Suban ahora el archivo two_clusters.csv")
        uploaded = files.upload()
        if "two_clusters.csv" not in uploaded:
            raise FileNotFoundError("No se subió two_clusters.csv.")
        data_path = Path("two_clusters.csv")
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontró two_clusters.csv. Colóquenlo junto al notebook."
        ) from exc

stars = pd.read_csv(data_path, dtype={"gaia_source_id": "string"})
required_columns = {
    "cluster", "gaia_source_id", "g_mag", "bp_mag", "rp_mag",
    "membership_probability", "use_star", "distance_pc",
    "a_g_mag", "e_bp_rp_mag",
}
missing = required_columns.difference(stars.columns)
if missing:
    raise ValueError(f"Faltan columnas: {sorted(missing)}")
if set(stars["cluster"].dropna()) != set(CLUSTERS):
    raise ValueError("El archivo no contiene exactamente los dos cúmulos esperados.")

print(f"Archivo cargado: {data_path}")
print(f"Filas disponibles: {len(stars):,}")
display(stars.head(3))

In [ ]:
# Recuperación de los cálculos estudiados en los días 1 y 2.
# No se calcula aquí el turn-off ni se anticipa la conclusión.
stars["bp_minus_rp"] = stars["bp_mag"] - stars["rp_mag"]
stars["distance_modulus"] = 5 * np.log10(stars["distance_pc"] / 10)
stars["corrected_colour"] = stars["bp_minus_rp"] - stars["e_bp_rp_mag"]
stars["absolute_g"] = (
    stars["g_mag"] - stars["distance_modulus"] - stars["a_g_mag"]
)
selected = stars.loc[stars["use_star"]].copy()

distance_results = (stars.groupby("cluster").agg(distance_pc=("distance_pc", "first"),
        distance_modulus=("distance_modulus", "first"),).reindex(CLUSTERS))
print(f"Días anteriores recuperados: {len(selected)} estrellas.")
display(distance_results.round(3))

In [ ]:
# Observen nuevamente el patrón antes de aplicar regiones numéricas.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)
for ax, cluster in zip(axes, CLUSTERS):
    group = selected.loc[selected["cluster"] == cluster]
    ax.scatter(
        group["corrected_colour"], group["absolute_g"],
        s=24, alpha=0.70, color=COLORS[cluster], edgecolor="none",
    )
    ax.set_title(LABELS[cluster])
    ax.set_xlabel(r"Color corregido $(G_{BP}-G_{RP})_0$ (mag)")
    ax.set_xlim(-0.5, 3.6)
    ax.set_ylim(12.8, -1.2)
axes[0].set_ylabel(r"Magnitud absoluta corregida $M_{G,0}$ (mag)")
fig.tight_layout()
plt.show()

<div style="width:100%; margin:5px 0 0px 0; padding:0px 0 0px 0px;
            font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:18px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    5. ¿Dónde termina cada secuencia principal?
  </div>
</div>

Primero señalen visualmente el turn-off. Luego utilicen las regiones autor-probadas:

| Cúmulo | Color corregido $C_0$ | Magnitud $M_{G,0}$ |
|---|---|---|
| Pléyades | $-0.10\le C_0\le0.10$ | $-0.50\le M_{G,0}\le1.30$ |
| Messier 67 | $0.55\le C_0\le0.85$ | $2.60\le M_{G,0}\le3.30$ |

Las cajas no crean el turn-off: resumen de manera reproducible una sección poblada del patrón. Usaremos la **mediana** para evitar que una sola estrella extrema decida la conclusión.

In [ ]:
# TAREA 6: construyan las dos selecciones rectangulares.
# Unan tres condiciones con & y usen .between(mínimo, máximo).
pleiades_turnoff = None
messier67_turnoff = None

if not isinstance(pleiades_turnoff, pd.DataFrame):
    raise ValueError("Construyan pleiades_turnoff.")
if not isinstance(messier67_turnoff, pd.DataFrame):
    raise ValueError("Construyan messier67_turnoff.")
if min(len(pleiades_turnoff), len(messier67_turnoff)) < 3:
    raise ValueError("Una región tiene muy pocas estrellas.")

print("Estrellas dentro de cada región:")
print(f"  Pléyades:   {len(pleiades_turnoff)}")
print(f"  Messier 67: {len(messier67_turnoff)}")

**Pista de sintaxis — úsela solo si la necesitan**

```python
region = selected.loc[
    (selected["cluster"] == "Nombre")
    & selected["corrected_colour"].between(color_min, color_max)
    & selected["absolute_g"].between(magnitud_min, magnitud_max)].copy()
```

In [ ]:
# TAREA 7: calculen color y magnitud medianos de cada región.
pleiades_median_colour = None
pleiades_median_magnitude = None
m67_median_colour = None
m67_median_magnitude = None

summary_values = [pleiades_median_colour, pleiades_median_magnitude,
    m67_median_colour, m67_median_magnitude,]
if any(value is None for value in summary_values):
    raise ValueError("Completen las cuatro medianas.")
if not np.isfinite(summary_values).all():
    raise ValueError("Alguna mediana no es finita.")

turnoff_results = pd.DataFrame({
    "n_turnoff": [len(pleiades_turnoff), len(messier67_turnoff)],
    "median_turnoff_colour": [pleiades_median_colour, m67_median_colour],
    "median_turnoff_absolute_g": [pleiades_median_magnitude, m67_median_magnitude],
}, index=CLUSTERS)
display(turnoff_results.round(3))

In [ ]:
# TAREA 8: calculen M67 − Pléyades.
delta_turnoff = None

if delta_turnoff is None:
    raise ValueError("Calculen delta_turnoff.")
print(f"Diferencia M67 − Pléyades: {delta_turnoff:.3f} mag")
print("Una magnitud mayor significa menor brillo intrínseco.")

In [ ]:
TURN_OFF_BOXES = {"Pleiades": (-0.10, 0.10, -0.50, 1.30), "Messier 67": (0.55, 0.85, 2.60, 3.30),}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)
for ax, cluster in zip(axes, CLUSTERS):
    group = selected.loc[selected["cluster"] == cluster]
    c_min, c_max, m_min, m_max = TURN_OFF_BOXES[cluster]
    result = turnoff_results.loc[cluster]
    ax.scatter(
        group["corrected_colour"], group["absolute_g"],
        s=22, alpha=0.58, color=COLORS[cluster], edgecolor="none",
        label="Estrellas seleccionadas",
    )
    ax.add_patch(Rectangle(
        (c_min, m_min), c_max - c_min, m_max - m_min,
        fill=False, linewidth=2.4, edgecolor="#9B51E0",
        label="Región de turn-off",
    ))
    ax.scatter(
        result["median_turnoff_colour"], result["median_turnoff_absolute_g"],
        marker="*", s=230, color="#EB5757", edgecolor="black",
        linewidth=0.7, zorder=5, label="Mediana",
    )
    ax.set_title(f"{LABELS[cluster]} (n = {int(result['n_turnoff'])})")
    ax.set_xlabel(r"Color corregido $(G_{BP}-G_{RP})_0$ (mag)")
    ax.set_xlim(-0.5, 3.6)
    ax.set_ylim(12.8, -1.2)
    ax.legend(loc="lower left", fontsize=9)

axes[0].set_ylabel(r"Magnitud absoluta corregida $M_{G,0}$ (mag)")
fig.suptitle("Figura 2. Turn-off representativo", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("figure_2_corrected_turnoff_cmds.png", dpi=220, bbox_inches="tight")
plt.show()
print("Figura guardada como figure_2_corrected_turnoff_cmds.png")

### Una complicación científica

Messier 67 puede mostrar algunas estrellas azules o brillantes por encima de su turn-off poblado. Podrían incluir *blue stragglers*, binarias no resueltas o miembros inciertos. No intenten clasificarlas individualmente.

> La edad se infiere de la secuencia poblada, no de la estrella más brillante o más azul.

<div style="width:100%; margin:5px 0 0px 0; padding:0px 0 0px 0px;
            font-family:Arial, Helvetica, sans-serif;">

  <div style="font-size:18px; font-weight:700; letter-spacing:0.14em;
              text-transform:uppercase; color:#526b84; margin-bottom:7px;">
    6. ¿ Qué cúmulo es más antiguo?
  </div>
</div>

Construyan la cadena de razonamiento:

1. Las estrellas más masivas son brillantes y consumen combustible rápidamente.
2. Abandonan antes la secuencia principal.
3. Un cúmulo viejo ya no conserva la misma población masiva y brillante en su secuencia ordinaria.
4. Su turn-off poblado aparece a una magnitud intrínseca mayor, es decir, más débil.

In [ ]:
final_results = distance_results.join(turnoff_results)
final_results.index = [LABELS[name] for name in final_results.index]
display(final_results.round(3))
print(f"ΔM_TO (Messier 67 − Pléyades) = {delta_turnoff:.3f} mag")

final_results.to_csv("project_results.csv", index_label="cluster")
print("Tabla guardada como project_results.csv")

### Conclusión del equipo

Redacten 100–150 palabras e incluyan:

- cuál cúmulo concluyen que es más antiguo;
- los dos módulos de distancia;
- las dos magnitudes medianas de turn-off;
- el signo y valor de la diferencia M67 − Pléyades;
- la explicación física que conecta un turn-off más débil con mayor edad.

> **Conclusión:** escriban aquí el párrafo.

### Limitación

Elijan una: extinción común aproximada, pertenencia estadística, binarias no resueltas, cajas simplificadas, *blue stragglers* o diferencias de composición química.

> **Limitación:** escriban aquí dos o tres oraciones.

**Checkpoint final:** la conclusión cita valores, interpreta correctamente las magnitudes e incluye una limitación.

## Lista de entrega

- `figure_1_apparent_cmds.png`, obtenida el día 1;
- `figure_2_corrected_turnoff_cmds.png`;
- `project_results.csv`;
- conclusión de 100–150 palabras;
- una limitación científica.

Antes de terminar, cada integrante debe responder: **¿qué cambió entre la primera y la segunda figura, y por qué ese cambio permite comparar edades?**

## Extensión opcional — solo con autorización del mentor

Si ya completaron todos los productos, prueben una muestra más estricta con `membership_probability >= 0.9`. Comparen si quedan menos estrellas, si la secuencia parece más limpia y si cambia la conclusión relativa.

In [ ]:
RUN_OPTIONAL_EXTENSION = False

if RUN_OPTIONAL_EXTENSION:
    # Añadan la condición de probabilidad a use_star.
    stricter_sample = None
    if stricter_sample is None:
        raise ValueError("Completen stricter_sample.")
    strict_counts = (
        stars.loc[stricter_sample].groupby("cluster").size().reindex(CLUSTERS)
    )
    display(strict_counts.rename("Selección más estricta").to_frame())
else:
    print("Extensión opcional desactivada.")

## Alcance

Los datos proceden del catálogo de cúmulos abiertos Hunt & Reffert (2024), CDS/VizieR `J/A+A/686/A42`, y utilizan fotometría Gaia.